In [ ]:
%run "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_01_Bronze/Jupyter Notebooks/00_bronze_data_connector.ipynb"


25/12/07 20:48:17 WARN Utils: Your hostname, Manojrams-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.0.219 instead (on interface en0)
25/12/07 20:48:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/manojrammopati/.ivy2/cache
The jars for the packages stored in: /Users/manojrammopati/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-azure added as a dependency
com.azure#azure-storage-blob added as a dependency
com.azure#azure-identity added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8a5f78a3-1ad1-4807-a1d4-0f033acb5390;1.0
	confs: [default]


:: loading settings :: url = jar:file:/opt/anaconda3/envs/spark_local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-azure;3.3.4 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found com.microsoft.azure#azure-storage;7.0.1 in central
	found com.fasterxml.jackson.core#jackson-core;2.12.7 in central
	found org.slf4j#slf4j-api;1.7.36 in central
	found com.microsoft.azure#azure-keyvault-core;1.0.0 in central
	found com.google.guava#guava;27.0-jre in central
	found com.google.guava#failureaccess;1.0 in central
	found com.google.guava#listenablefuture;9999.0-empty-to-avoid-conflict-with-guava in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found org.checkerframework#checker-qual;2.5.2 in central
	found com.google.errorpron

Preconnectors ran successfully and connected to 'clientdatastorage' storage accounts
 'policy_df, claim_df, member_df, provider_df' are ready to use from container 'raw data'(meridian architecture of bronze) 


# Cell 1 – Intro markdown

ML – Claims Risk Model (Fraud / High Cost)

**Goal:** Train and evaluate Spark ML models to predict *claims risk* using the curated
 Gold feature table `ft_claims_risk_split`.

We focus on:
- Label: `Is_Fraudulent_Claim` (binary 0/1)
- Models: Random Forest, Logistic Regression, Gradient Boosted Trees

 This notebook mirrors the structure of the policy churn ML notebook:
 1. Load Gold feature table
 2. Clean + split (train / test)
 3. Build Spark ML pipeline (indexing, OHE, assembler)
 4. Train several models and compare metrics
 5. Persist the best model to ADLS
#6. Score sample claims to show how it would work in production


# Cell 2 – Imports & configuration

In [3]:
# Cell 2 — Imports & config

from pyspark.sql import functions as F
from pyspark.sql import DataFrame

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import (
    RandomForestClassifier,
    LogisticRegression,
    GBTClassifier
)
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array

# Storage / paths (adjust only if your account or container names differ)
ACCOUNT       = "clientdatastorage"
GOLD_CONTAINER = "golddata"

FT_CLAIMS_RISK_SPLIT_PATH = (
    f"abfss://{GOLD_CONTAINER}@{ACCOUNT}.dfs.core.windows.net/ft_claims_risk_split"
)

MODEL_BASE_PATH = (
    f"abfss://{GOLD_CONTAINER}@{ACCOUNT}.dfs.core.windows.net/models"
)
MODEL_NAME      = "claims_risk_best"
MODEL_PATH      = f"{MODEL_BASE_PATH}/{MODEL_NAME}"

print("Feature table path:", FT_CLAIMS_RISK_SPLIT_PATH)
print("Model path        :", MODEL_PATH)


Feature table path: abfss://golddata@clientdatastorage.dfs.core.windows.net/ft_claims_risk_split
Model path        : abfss://golddata@clientdatastorage.dfs.core.windows.net/models/claims_risk_best


# Cell 3 – Load feature table & basic checks

In [4]:
# Cell 3 — Load ft_claims_risk_split and inspect

claims_all = (
    spark.read
         .format("delta")
         .load(FT_CLAIMS_RISK_SPLIT_PATH)
)

print("Total rows in ft_claims_risk_split:", claims_all.count())
claims_all.printSchema()
claims_all.show(5, truncate=False)


25/12/07 20:48:28 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Total rows in ft_claims_risk_split: 558211
root
 |-- Claim_ID: string (nullable = true)
 |-- Member_Key: string (nullable = true)
 |-- Provider_ID: string (nullable = true)
 |-- Claim_Type_Name: string (nullable = true)
 |-- Claim_Type_Code: string (nullable = true)
 |-- Claim_Status: string (nullable = true)
 |-- Claim_Amount_GBP: double (nullable = true)
 |-- Payout_GBP: double (nullable = true)
 |-- Payout_to_Amount_Ratio: double (nullable = true)
 |-- Days_To_Settle: integer (nullable = true)
 |-- High_Cost_Claim_Flag: integer (nullable = true)
 |-- Claim_Fraud_Label: integer (nullable = true)
 |-- Provider_Fraud_Label: integer (nullable = true)
 |-- Provider_Risk_Tier: string (nullable = true)
 |-- dq_money_valid: integer (nullable = true)
 |-- dq_date_valid: integer (nullable = true)
 |-- Is_Fraudulent_Claim: integer (nullable = true)
 |-- Is_High_Cost: integer (nullable = true)
 |-- dataset_split: string (nullable = true)



+---------+----------+-----------+---------------+---------------+------------+------------------+----------+----------------------+--------------+--------------------+-----------------+--------------------+------------------+--------------+-------------+-------------------+------------+-------------+
|Claim_ID |Member_Key|Provider_ID|Claim_Type_Name|Claim_Type_Code|Claim_Status|Claim_Amount_GBP  |Payout_GBP|Payout_to_Amount_Ratio|Days_To_Settle|High_Cost_Claim_Flag|Claim_Fraud_Label|Provider_Fraud_Label|Provider_Risk_Tier|dq_money_valid|dq_date_valid|Is_Fraudulent_Claim|Is_High_Cost|dataset_split|
+---------+----------+-----------+---------------+---------------+------------+------------------+----------+----------------------+--------------+--------------------+-----------------+--------------------+------------------+--------------+-------------+-------------------+------------+-------------+
|CLM110013|BENE15435 |PRV51790   |Travel         |TRAVEL         |Settled     |244.07511199

# Cell 4 – Filter label + train/test split

We’ll model fraud risk first: Is_Fraudulent_Claim as label.

In [5]:
# Cell 4 — Filter label != null and create train/test splits

LABEL_COL = "Is_Fraudulent_Claim"

if LABEL_COL not in claims_all.columns:
    raise ValueError(f"Expected label column '{LABEL_COL}' not found in ft_claims_risk_split")

# Only keep rows where label is 0 or 1
claims_labeled = (
    claims_all
      .filter(F.col(LABEL_COL).isNotNull())
      .withColumn(LABEL_COL, F.col(LABEL_COL).cast("double"))
)

# Optional: filter to dq-valid rows only, if you want
claims_labeled = claims_labeled.filter(
    (F.col("dq_money_valid") == 1) & (F.col("dq_date_valid") == 1)
)

print("Rows with non-null label and dq-valid:", claims_labeled.count())

# Train / test split from pre-assigned column
train_df = claims_labeled.filter(F.col("dataset_split") == "train")
test_df  = claims_labeled.filter(F.col("dataset_split") == "test")

print("Train rows:", train_df.count())
print("Test rows :", test_df.count())


Rows with non-null label and dq-valid: 558211
Train rows: 446102
Test rows : 112109


# Cell 5 – Define feature columns & preprocessing function

In [6]:
# Cell 5 — Define feature columns & preprocessing helpers

# Candidate numeric features (we'll only keep those that exist)
numeric_candidates = [
    "Claim_Amount_GBP",
    "Payout_GBP",
    "Payout_to_Amount_Ratio",
    "Days_To_Settle",
]

# Candidate categorical features
categorical_candidates = [
    "Claim_Type_Name",
    "Claim_Status",
    "Claim_Type_Code",
    "Provider_Risk_Tier",
    "Provider_ID",      # optional: treat provider_id as category
]

cols = claims_labeled.columns

numeric_cols = [c for c in numeric_candidates if c in cols]
cat_cols     = [c for c in categorical_candidates if c in cols]

print("Numeric features   :", numeric_cols)
print("Categorical features:", cat_cols)

# ID / context columns to keep for interpretability (not part of features vec)
id_cols = [c for c in ["Claim_ID", "Member_Key", "Provider_ID"] if c in cols]


def prep_nulls(df: DataFrame) -> DataFrame:
    """
    Simple null-handling: numeric -> 0, categorical -> 'Unknown'.
    """
    d = df
    for c in numeric_cols:
        d = d.withColumn(c, F.when(F.col(c).isNull(), F.lit(0.0)).otherwise(F.col(c)))
    for c in cat_cols:
        d = d.withColumn(c, F.when(F.col(c).isNull(), F.lit("Unknown")).otherwise(F.col(c)))
    return d


Numeric features   : ['Claim_Amount_GBP', 'Payout_GBP', 'Payout_to_Amount_Ratio', 'Days_To_Settle']
Categorical features: ['Claim_Type_Name', 'Claim_Status', 'Claim_Type_Code', 'Provider_Risk_Tier', 'Provider_ID']


# Cell 6 – Build the Spark ML pipeline skeleton

In [7]:
# Cell 6 — Build pipeline: indexers, encoders, assembler

# 1) StringIndexers for categoricals
indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=f"{c}_idx",
        handleInvalid="keep"
    )
    for c in cat_cols
]

# 2) OneHotEncoders
encoders = [
    OneHotEncoder(
        inputCol=f"{c}_idx",
        outputCol=f"{c}_oh",
        handleInvalid="keep"
    )
    for c in cat_cols
]

# 3) VectorAssembler for final features
feature_cols = numeric_cols + [f"{c}_oh" for c in cat_cols]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="keep",
)

# Note: classifier itself will be plugged in later (RF, LR, GBT).
print("Feature columns in assembler:", feature_cols)


Feature columns in assembler: ['Claim_Amount_GBP', 'Payout_GBP', 'Payout_to_Amount_Ratio', 'Days_To_Settle', 'Claim_Type_Name_oh', 'Claim_Status_oh', 'Claim_Type_Code_oh', 'Provider_Risk_Tier_oh', 'Provider_ID_oh']


# Cell 7 – Train & evaluate helper function

In [8]:
# Cell 7 — Train & evaluate helper

def train_and_eval(pipeline: Pipeline, train_df: DataFrame, test_df: DataFrame, model_name: str):
    print(f"\n===== Training {model_name} =====")
    model = pipeline.fit(train_df)

    print(f"Scoring test set with {model_name} ...")
    pred = model.transform(test_df)

    evaluator_roc = BinaryClassificationEvaluator(
        labelCol=LABEL_COL,
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC"
    )
    evaluator_pr = BinaryClassificationEvaluator(
        labelCol=LABEL_COL,
        rawPredictionCol="rawPrediction",
        metricName="areaUnderPR"
    )

    roc_auc = evaluator_roc.evaluate(pred)
    pr_auc  = evaluator_pr.evaluate(pred)

    # Confusion-matrix style metrics
    preds = (
        pred.select(
            LABEL_COL,
            F.col("prediction").cast("double").alias("prediction")
        )
    )

    tp = preds.filter((F.col(LABEL_COL) == 1) & (F.col("prediction") == 1)).count()
    tn = preds.filter((F.col(LABEL_COL) == 0) & (F.col("prediction") == 0)).count()
    fp = preds.filter((F.col(LABEL_COL) == 0) & (F.col("prediction") == 1)).count()
    fn = preds.filter((F.col(LABEL_COL) == 1) & (F.col("prediction") == 0)).count()

    total = tp + tn + fp + fn
    accuracy  = (tp + tn) / total if total else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall    = tp / (tp + fn) if (tp + fn) else 0.0
    f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

    metrics = {
        "model": model_name,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }

    print(f"ROC AUC: {roc_auc:.4f} | PR AUC: {pr_auc:.4f} | "
          f"Acc: {accuracy:.4f} | Prec: {precision:.4f} | Rec: {recall:.4f} | F1: {f1:.4f}")

    return model, metrics, pred


# Cell 8 – Define models & run training

In [10]:
# Cell 8 — Define classifiers and train them

# Prepare data with null-handling
train_pre = prep_nulls(train_df)
test_pre  = prep_nulls(test_df)

# Common classifier settings
rf = RandomForestClassifier(
    labelCol=LABEL_COL,
    featuresCol="features",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    numTrees=100,
    maxDepth=8,
    seed=42,
)

lr = LogisticRegression(
    labelCol=LABEL_COL,
    featuresCol="features",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    maxIter=50,
    regParam=0.01,
    elasticNetParam=0.0,
)
"""

gbt = GBTClassifier(
    labelCol=LABEL_COL,
    featuresCol="features",
    predictionCol="prediction",
    maxIter=80,
    maxDepth=6,
    stepSize=0.1,
    seed=42,
)
    """

pipelines = {
    "RandomForest": Pipeline(stages=indexers + encoders + [assembler, rf]),
    "LogisticRegression": Pipeline(stages=indexers + encoders + [assembler, lr]),
   # "GBTClassifier": Pipeline(stages=indexers + encoders + [assembler, gbt]),
}

results = []
trained_models = {}

for name, pipe in pipelines.items():
    model, metrics, _ = train_and_eval(pipe, train_pre, test_pre, name)
    results.append(metrics)
    trained_models[name] = model

print("\n===== Summary of models =====")
for m in results:
    print(m)



===== Training RandomForest =====


25/12/07 21:26:00 WARN MemoryStore: Not enough space to cache rdd_1050_2 in memory! (computed 321.9 MiB so far)
25/12/07 21:26:00 WARN BlockManager: Persisting block rdd_1050_2 to disk instead.
25/12/07 21:26:01 WARN MemoryStore: Not enough space to cache rdd_556_2 in memory! (computed 315.3 MiB so far)
25/12/07 21:26:19 WARN MemoryStore: Not enough space to cache rdd_556_2 in memory! (computed 315.3 MiB so far)
25/12/07 21:26:25 WARN MemoryStore: Not enough space to cache rdd_1050_2 in memory! (computed 321.9 MiB so far)
25/12/07 21:26:35 WARN MemoryStore: Not enough space to cache rdd_556_2 in memory! (computed 315.3 MiB so far)
25/12/07 21:26:46 WARN MemoryStore: Not enough space to cache rdd_1050_2 in memory! (computed 321.9 MiB so far)
25/12/07 21:26:50 WARN MemoryStore: Not enough space to cache rdd_556_2 in memory! (computed 315.3 MiB so far)
25/12/07 21:27:05 WARN MemoryStore: Not enough space to cache rdd_556_2 in memory! (computed 315.3 MiB so far)
25/12/07 21:27:06 WARN Memo

Scoring test set with RandomForest ...


25/12/07 21:29:42 WARN DAGScheduler: Broadcasting large task binary with size 1042.0 KiB
25/12/07 21:29:46 WARN DAGScheduler: Broadcasting large task binary with size 1042.0 KiB
25/12/07 21:29:53 WARN MemoryStore: Not enough space to cache rdd_556_2 in memory! (computed 315.3 MiB so far)


ROC AUC: 0.9872 | PR AUC: 0.9906 | Acc: 0.7461 | Prec: 1.0000 | Rec: 0.3029 | F1: 0.4650

===== Training LogisticRegression =====


25/12/07 21:30:10 WARN MemoryStore: Not enough space to cache rdd_556_2 in memory! (computed 315.3 MiB so far)


Scoring test set with LogisticRegression ...


ROC AUC: 0.9870 | PR AUC: 0.9890 | Acc: 0.9906 | Prec: 1.0000 | Rec: 0.9743 | F1: 0.9870

===== Summary of models =====
{'model': 'RandomForest', 'roc_auc': 0.9872447507712879, 'pr_auc': 0.9905508477073157, 'accuracy': 0.7461042378399593, 'precision': 1.0, 'recall': 0.30291675850415106, 'f1': 0.46498251945415586, 'tp': 12369, 'tn': 71276, 'fp': 0, 'fn': 28464}
{'model': 'LogisticRegression', 'roc_auc': 0.9870040565735424, 'pr_auc': 0.9890324066888636, 'accuracy': 0.9906251951226039, 'precision': 0.9999748642670421, 'recall': 0.9742855043714642, 'f1': 0.9869630474961857, 'tp': 39783, 'tn': 71275, 'fp': 1, 'fn': 1050}


25/12/07 21:30:28 WARN MemoryStore: Not enough space to cache rdd_556_2 in memory! (computed 315.3 MiB so far)
25/12/07 21:30:43 WARN MemoryStore: Not enough space to cache rdd_556_2 in memory! (computed 315.3 MiB so far)
25/12/07 21:30:58 WARN MemoryStore: Not enough space to cache rdd_556_2 in memory! (computed 315.3 MiB so far)


# Cell 9 – Choose best model & save to ADLS

In [11]:
# Cell 9 — Pick best model (by ROC AUC) and save it

# Pick the model with the highest ROC AUC
best = sorted(results, key=lambda r: r["roc_auc"], reverse=True)[0]
best_name = best["model"]
best_model = trained_models[best_name]

print(f"\nBest model by ROC AUC: {best_name} (ROC AUC={best['roc_auc']:.4f})")

# Save the whole PipelineModel to ADLS so scoring can reuse preprocessing + classifier
best_model.write().overwrite().save(MODEL_PATH)
print(f"✅ Saved best model to: {MODEL_PATH}")



Best model by ROC AUC: RandomForest (ROC AUC=0.9872)


25/12/07 21:31:14 WARN MemoryStore: Not enough space to cache rdd_556_2 in memory! (computed 315.3 MiB so far)
25/12/07 21:31:14 WARN MemoryStore: Not enough space to cache rdd_556_2 in memory! (computed 91.5 MiB so far)
25/12/07 21:31:14 WARN MemoryStore: Not enough space to cache rdd_556_2 in memory! (computed 35.4 MiB so far)
25/12/07 21:31:14 WARN MemoryStore: Not enough space to cache rdd_1110_2 in memory! (computed 3.7 MiB so far)
25/12/07 21:31:14 WARN MemoryStore: Failed to reserve initial memory threshold of 1024.0 KiB for computing block rdd_1436_2 in memory.
25/12/07 21:31:14 WARN MemoryStore: Not enough space to cache rdd_1436_2 in memory! (computed 384.0 B so far)
25/12/07 21:31:14 WARN BlockManager: Persisting block rdd_1436_2 to disk instead.


✅ Saved best model to: abfss://golddata@clientdatastorage.dfs.core.windows.net/models/claims_risk_best


25/12/07 21:31:59 WARN MemoryStore: Not enough space to cache rdd_556_2 in memory! (computed 315.3 MiB so far)
25/12/07 21:32:16 WARN MemoryStore: Not enough space to cache rdd_556_2 in memory! (computed 315.3 MiB so far)


# Cell 10 – Load model & score sample claims

In [12]:
# Cell 10 — Load persisted model and score sample claims

loaded_model = PipelineModel.load(MODEL_PATH)

# Take a random sample of test claims
sample_to_score = (
    test_df
        .orderBy(F.rand())
        .limit(20)
)

scored_raw = loaded_model.transform(prep_nulls(sample_to_score))

# Convert probability vector to array and take probability of class 1 (fraud)
scored = (
    scored_raw
        .withColumn("prob_arr", vector_to_array("probability"))
        .withColumn("fraud_prob", F.col("prob_arr")[1])
        .select(
            *[c for c in id_cols if c in scored_raw.columns],
            LABEL_COL,
            "fraud_prob",
            "prediction",
            "Claim_Type_Name",
            "Claim_Status",
            "Claim_Amount_GBP",
            "Payout_GBP",
            "Payout_to_Amount_Ratio",
            "Provider_Risk_Tier",
        )
)

scored.printSchema()
scored.show(truncate=False)


25/12/07 21:32:33 WARN MemoryStore: Not enough space to cache rdd_556_2 in memory! (computed 315.3 MiB so far)
25/12/07 21:32:48 WARN MemoryStore: Not enough space to cache rdd_556_2 in memory! (computed 315.3 MiB so far)


root
 |-- Claim_ID: string (nullable = true)
 |-- Member_Key: string (nullable = true)
 |-- Provider_ID: string (nullable = true)
 |-- Is_Fraudulent_Claim: double (nullable = true)
 |-- fraud_prob: double (nullable = true)
 |-- prediction: double (nullable = false)
 |-- Claim_Type_Name: string (nullable = true)
 |-- Claim_Status: string (nullable = true)
 |-- Claim_Amount_GBP: double (nullable = true)
 |-- Payout_GBP: double (nullable = true)
 |-- Payout_to_Amount_Ratio: double (nullable = true)
 |-- Provider_Risk_Tier: string (nullable = true)



+---------+----------+-----------+-------------------+------------------+----------+---------------+------------+------------------+----------+----------------------+------------------+
|Claim_ID |Member_Key|Provider_ID|Is_Fraudulent_Claim|fraud_prob        |prediction|Claim_Type_Name|Claim_Status|Claim_Amount_GBP  |Payout_GBP|Payout_to_Amount_Ratio|Provider_Risk_Tier|
+---------+----------+-----------+-------------------+------------------+----------+---------------+------------+------------------+----------+----------------------+------------------+
|CLM701428|BENE139222|PRV55004   |1.0                |0.5294912170236015|1.0       |Outpatient     |Settled     |686.0567634494677 |500.0     |0.7288026685809769    |High risk         |
|CLM615611|BENE86285 |PRV57287   |0.0                |0.2965140561292054|0.0       |Hospital       |Settled     |79.74476532311047 |60.0      |0.7524004836792926    |Low risk          |
|CLM520574|BENE28926 |PRV57192   |0.0                |0.29651405612920

25/12/07 21:33:03 WARN MemoryStore: Not enough space to cache rdd_556_2 in memory! (computed 315.3 MiB so far)
25/12/07 21:33:20 WARN MemoryStore: Not enough space to cache rdd_556_2 in memory! (computed 315.3 MiB so far)
25/12/07 21:33:20 WARN MemoryStore: Not enough space to cache rdd_556_2 in memory! (computed 91.5 MiB so far)
25/12/07 21:33:20 WARN MemoryStore: Not enough space to cache rdd_556_2 in memory! (computed 35.4 MiB so far)
25/12/07 21:33:20 WARN MemoryStore: Not enough space to cache rdd_1436_2 in memory! (computed 3.7 MiB so far)
25/12/07 21:33:20 WARN MemoryStore: Failed to reserve initial memory threshold of 1024.0 KiB for computing block rdd_1693_2 in memory.
25/12/07 21:33:20 WARN MemoryStore: Not enough space to cache rdd_1693_2 in memory! (computed 384.0 B so far)
25/12/07 21:33:20 WARN BlockManager: Persisting block rdd_1693_2 to disk instead.
25/12/07 21:34:07 WARN MemoryStore: Not enough space to cache rdd_556_2 in memory! (computed 315.3 MiB so far)
25/12/07 2